# 01 · Agent Loop básico

Objetivo: enseñar el punto de partida **antes** de tener un harness completo.

Este notebook muestra la diferencia entre:

1. preparar un prompt,
2. hacer una llamada simple al modelo si la configuración local está disponible,
3. usar una respuesta simulada como fallback reproducible,
4. reconocer qué falta para convertirlo en un agente operativo.

Tiempo estimado: **1 minuto**.

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

prompt = 'Explícame Agent Harness en 5 bullets para una audiencia técnica.'
print('1) Prompt preparado para el modelo:')
print(prompt)

1) Prompt preparado para el modelo:
Explícame Agent Harness en 5 bullets para una audiencia técnica.


## Llamada simple al modelo

Esta sección intenta llamar directamente al cliente de chat configurado en el proyecto. No registra tools, no lee contexto y no persiste sesión: es solo modelo + prompt.

Para no mezclar infraestructura con el concepto que queremos mostrar, primero hay un paso previo de preparación del entorno. La celda importante para narrar es la siguiente: `build_chat_client()` + `client.get_response(...)`.

Si faltan credenciales o variables de entorno, la llamada no falla la demo; pasa al ejemplo simulado de abajo.

### Paso previo: cargar configuración local

Este bloque es preparación de demo. Carga en el kernel de Jupyter las mismas variables que `scripts/load_model_from_akv.sh` carga en una terminal, sin imprimir secretos.

In [ ]:
import json
import os
import subprocess
import warnings

warnings.filterwarnings('ignore', message='.*experimental.*', category=Warning)

MODEL_ENV_KEYS = [
    'AZURE_OPENAI_BASE_URL',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_MODEL',
    'AZURE_OPENAI_CHAT_COMPLETION_MODEL',
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_API_VERSION',
    'AZURE_OPENAI_MAX_COMPLETION_TOKENS',
]


def load_model_env_from_script() -> bool:
    script = ROOT / 'scripts' / 'load_model_from_akv.sh'
    if not script.exists():
        return False

    command = f'''
set -euo pipefail
source "{script}" >/dev/null
python - <<'PY'
import json
import os

keys = {MODEL_ENV_KEYS!r}
print(json.dumps({{key: os.environ[key] for key in keys if os.environ.get(key)}}))
PY
'''
    completed = subprocess.run(
        ['bash', '-lc', command],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    values = json.loads(completed.stdout)
    os.environ.update(values)
    return bool(values)

if load_model_env_from_script():
    print('Configuración de modelo cargada en el kernel del notebook.')
    print(f"Modelo: {os.environ.get('AZURE_OPENAI_CHAT_COMPLETION_MODEL')}")
else:
    print('No se encontró scripts/load_model_from_akv.sh; se usará la configuración ya presente en el entorno.')

In [ ]:
from agent_framework import Message  # noqa: E402
from microharness.harness import build_chat_client  # noqa: E402

try:
    client = build_chat_client()
    response = await client.get_response([Message(role='user', contents=[prompt])])
    model_answer = response.text.strip()
    print('2) Respuesta real del modelo:')
    print(model_answer)
except Exception as exc:
    model_answer = None
    print('2) No se pudo llamar al modelo real en este entorno.')
    print(f'Motivo: {type(exc).__name__}: {exc}')
    print('Continúa con la respuesta simulada de la siguiente celda.')

2) Configuración de modelo cargada en el kernel del notebook.
Modelo: gpt-5-5-thinking

3) Respuesta real del modelo:
- **Agent Harness es una capa de ejecución para agentes**: encapsula cómo un agente recibe una tarea, invoca un modelo, usa herramientas externas, mantiene estado y produce una respuesta o acción.

- **Orquesta el ciclo de vida del agente**: gestiona pasos como planificación, llamadas al LLM, ejecución de tools/functions, validación de resultados, retries, timeouts y manejo de errores.

- **Estandariza la integración con herramientas**: permite exponer APIs, bases de datos, sistemas internos o funciones como “tools” que el agente puede invocar de forma controlada, normalmente con esquemas de entrada/salida bien definidos.

- **Facilita observabilidad y control**: registra trazas, prompts, llamadas a herramientas, latencias, costes, decisiones intermedias y errores, lo que permite debugging, evaluación y auditoría.

- **Sirve como base para producción**: añade capacidade

## Fallback simulado

Esta celda mantiene la demo reproducible incluso sin red, credenciales o coste de modelo.

La idea es ver el límite de una llamada básica: puede haber texto de salida, real o simulado, pero todavía no hay tools, contexto controlado, sesión ni endpoint.

In [6]:
simulated_model_answer = '''
Un modelo puede razonar y generar texto, pero por sí solo no sabe qué fuentes consultar,
qué acciones están permitidas ni cómo persistir estado.
'''

if globals().get('model_answer'):
    print('4) Fallback simulado omitido porque ya hubo respuesta real del modelo.')
else:
    print('4) Respuesta simulada del modelo:')
    print(simulated_model_answer.strip())

print('\n5) Conclusión: esto aún NO es un harness; es solo una llamada básica al modelo.')

4) Fallback simulado omitido porque ya hubo respuesta real del modelo.

5) Conclusión: esto aún NO es un harness; es solo una llamada básica al modelo.


## Mensaje para narrar

> Ahora sí hay una llamada simple al modelo cuando la configuración local está disponible. Pero sigue siendo solo modelo + prompt: todavía no es un agente operativo.

En los siguientes notebooks se añade lo que falta:

- tools,
- contexto controlado,
- estado de sesión,
- servidor FastAPI,
- streaming AG-UI.